<a href="https://colab.research.google.com/github/lankamokshagna-cyber/lankamokshagna-cyber.github.io/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image
import scipy.ndimage as ndimage

# ---------------------------------------------------------
# 1. Feature Extraction Functions
# ---------------------------------------------------------
def get_frequency_features(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (224, 224))
    f_transform = np.fft.fft2(img)
    f_shift = np.fft.fftshift(f_transform)
    magnitude_spectrum = 20 * np.log(np.abs(f_shift) + 1e-8)
    # Convert to tensor and add a channel dimension (1, 224, 224)
    return torch.tensor(magnitude_spectrum, dtype=torch.float32).unsqueeze(0)

def extract_noise_residual(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (224, 224)).astype(np.float32)
    # High-pass filter to extract camera noise
    kernel = np.array([[-1, -1, -1], [-1,  8, -1], [-1, -1, -1]]) / 8.0
    noise_residual = ndimage.convolve(img, kernel)
    return torch.tensor(noise_residual, dtype=torch.float32).unsqueeze(0)

# ---------------------------------------------------------
# 2. The Custom PyTorch Dataset
# ---------------------------------------------------------
class DeepfakeMultiModalDataset(Dataset):
    def __init__(self, csv_file, split_name):
        # Load the CSV your friend's script made
        self.data = pd.read_csv(csv_file)
        # Filter for train, val, or test
        self.data = self.data[self.data['split'] == split_name].reset_index(drop=True)

        # Standard Transformer Image Prep
        self.spatial_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = self.data.loc[idx, 'path']
        label = self.data.loc[idx, 'label'] # 0 for Real, 1 for Fake

        # 1. Spatial Arm (RGB Image)
        img_pil = Image.open(img_path).convert('RGB')
        spatial_data = self.spatial_transform(img_pil)

        # 2. Frequency Arm (FFT)
        freq_data = get_frequency_features(img_path)

        # 3. Prior Arm (Noise)
        noise_data = extract_noise_residual(img_path)

        return spatial_data, freq_data, noise_data, torch.tensor(label, dtype=torch.long)

print("✅ Data Loader Ready!")

✅ Data Loader Ready!


In [31]:
import torchvision.models as models

class MultiModalDeepfakeDetector(nn.Module):
    def __init__(self):
        super(MultiModalDeepfakeDetector, self).__init__()

        # ==========================================
        # ARM 1: Spatial (Vision Transformer)
        # ==========================================
        # We use a pre-trained ViT. It expects 3-channel RGB images.
        self.transformer_arm = models.vit_b_16(pretrained=True)
        # Change the final layer to output exactly 256 features instead of 1000 classes
        self.transformer_arm.heads.head = nn.Linear(self.transformer_arm.heads.head.in_features, 256)

        # ==========================================
        # ARM 2: Frequency (CNN)
        # ==========================================
        # A simple CNN to look for checkerboard patterns in the FFT map
        self.frequency_arm = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), # 1 input channel (Grayscale)
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(32 * 56 * 56, 128) # Condense to 128 features
        )

        # ==========================================
        # ARM 3: Noise Priors (CNN)
        # ==========================================
        # A simple CNN to look for disrupted sensor noise
        self.noise_arm = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(32 * 56 * 56, 128) # Condense to 128 features
        )

        # ==========================================
        # THE HEAD: Merging it all together
        # ==========================================
        # 256 (Transformer) + 128 (Freq) + 128 (Noise) = 512 total features
        self.classifier = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.5), # Prevents overfitting
            nn.Linear(128, 2) # Final Output: 2 Classes (Real or Fake)
        )

    def forward(self, spatial, freq, noise):
        # 1. Pass data through their respective arms
        spatial_features = self.transformer_arm(spatial)
        freq_features = self.frequency_arm(freq)
        noise_features = self.noise_arm(noise)

        # 2. Concatenate (glue) the features together in a straight line
        combined_features = torch.cat((spatial_features, freq_features, noise_features), dim=1)

        # 3. Make the final prediction
        output = self.classifier(combined_features)
        return output

print("✅ Multi-Modal Architecture Ready!")

✅ Multi-Modal Architecture Ready!


In [32]:
# 1. Define where the CSV is located (from your friend's script output)
CSV_PATH = '/content/processed_dataset/dataset_manifest.csv'

# 2. Create the PyTorch DataLoaders (this feeds images in batches of 16 so the GPU doesn't crash)
train_dataset = DeepfakeMultiModalDataset(CSV_PATH, split_name='train')
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

val_dataset = DeepfakeMultiModalDataset(CSV_PATH, split_name='val')
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

# 3. Initialize the Model and move it to the GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = MultiModalDeepfakeDetector().to(device)

# 4. Set the Loss Function and Optimizer
criterion = nn.CrossEntropyLoss() # Standard for classification
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001) # Adam is highly effective for Transformers

print("✅ Engine is primed. Ready for training loop!")

Using device: cuda
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:01<00:00, 206MB/s]


✅ Engine is primed. Ready for training loop!


In [37]:
import time
import copy

# ==========================================
# THE UPGRADED TRAINING LOOP
# ==========================================
# We can safely increase this to 10 now because the scheduler will stop it from going crazy!
EPOCHS = 10

# ---------------------------------------
# NEW UPGRADES INITIALIZED HERE
# ---------------------------------------
# 1. The Learning Rate Scheduler:
# It watches the Validation Loss. If the loss stops dropping for 2 epochs (patience=2),
# it cuts the learning rate in half (factor=0.5) so the AI takes smaller, more careful steps.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

# 2. The "Best Model" Tracker:
best_val_acc = 0.0
best_model_weights = copy.deepcopy(model.state_dict()) # Holds the smartest brain in temporary memory

print("🚀 Starting Upgraded Training Process...")

for epoch in range(EPOCHS):
    start_time = time.time()

    # ---------------------------------------
    # 1. TRAINING PHASE
    # ---------------------------------------
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for spatial, freq, noise, labels in train_loader:
        spatial, freq, noise, labels = spatial.to(device), freq.to(device), noise.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(spatial, freq, noise)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * spatial.size(0)
        _, predicted = torch.max(outputs.data, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    epoch_train_loss = train_loss / train_total
    epoch_train_acc = (train_correct / train_total) * 100

    # ---------------------------------------
    # 2. VALIDATION PHASE
    # ---------------------------------------
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for spatial, freq, noise, labels in val_loader:
            spatial, freq, noise, labels = spatial.to(device), freq.to(device), noise.to(device), labels.to(device)

            outputs = model(spatial, freq, noise)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * spatial.size(0)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    epoch_val_loss = val_loss / val_total
    epoch_val_acc = (val_correct / val_total) * 100

    end_time = time.time()

    # ---------------------------------------
    # 3. APPLYING THE UPGRADES
    # ---------------------------------------
    print(f"Epoch [{epoch+1}/{EPOCHS}] | Time: {end_time - start_time:.0f}s")
    print(f"   Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}%")
    print(f"   Val Loss:   {epoch_val_loss:.4f} | Val Acc:   {epoch_val_acc:.2f}%")

    # Upgrade A: Tell the scheduler to check the Val Loss
    scheduler.step(epoch_val_loss)

    # Upgrade B: Check if this epoch is the smartest one yet
    if epoch_val_acc > best_val_acc:
        print(f"   🌟 New Best Validation Accuracy! ({best_val_acc:.2f}% --> {epoch_val_acc:.2f}%) Saving brain...")
        best_val_acc = epoch_val_acc
        best_model_weights = copy.deepcopy(model.state_dict())

    print("-" * 50)

# ==========================================
# 4. FINAL WRAP UP
# ==========================================
# Load the absolute best weights back into the active model
model.load_state_dict(best_model_weights)

print(f"🎉 Training Complete! The smartest version of the model (Val Acc: {best_val_acc:.2f}%) was preserved.")
torch.save(model.state_dict(), '/content/best_deepfake_multimodal_model.pth')
print("💾 Model saved as 'best_deepfake_multimodal_model.pth'")

🚀 Starting Upgraded Training Process...
Epoch [1/10] | Time: 223s
   Train Loss: 0.3021 | Train Acc: 88.01%
   Val Loss:   0.3231 | Val Acc:   87.83%
   🌟 New Best Validation Accuracy! (0.00% --> 87.83%) Saving brain...
--------------------------------------------------
Epoch [2/10] | Time: 234s
   Train Loss: 0.2874 | Train Acc: 87.89%
   Val Loss:   0.2920 | Val Acc:   88.06%
   🌟 New Best Validation Accuracy! (87.83% --> 88.06%) Saving brain...
--------------------------------------------------
Epoch [3/10] | Time: 234s
   Train Loss: 0.2773 | Train Acc: 88.44%
   Val Loss:   0.3040 | Val Acc:   87.95%
--------------------------------------------------
Epoch [4/10] | Time: 234s
   Train Loss: 0.2531 | Train Acc: 89.33%
   Val Loss:   0.3076 | Val Acc:   87.17%
--------------------------------------------------
Epoch [5/10] | Time: 233s
   Train Loss: 0.2033 | Train Acc: 91.58%
   Val Loss:   0.3063 | Val Acc:   88.28%
   🌟 New Best Validation Accuracy! (88.06% --> 88.28%) Saving bra